# Geospatial Analysis — Station Cost Map

Visualises gas stations in geographic space, coloured by **effective cost** rather than price per gallon.

This shows the spatial decision landscape: which stations in the vicinity actually give you the best deal once you account for the fuel burned driving to them.

In [ ]:
import sys, os

# Works whether Jupyter is launched from Gas App/ or Gas App/notebooks/
_cwd = os.getcwd()
_root = _cwd if os.path.exists(os.path.join(_cwd, 'costcalc.py')) else os.path.dirname(_cwd)
sys.path.insert(0, _root)
print(f'Project root: {_root}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

from costcalc import VehicleParams, Station, calculate_station_result, rank_stations

plt.style.use('seaborn-v0_8-white')
plt.rcParams.update({'figure.dpi': 120})
print('Setup complete.')

## Synthetic Station Generator

Places stations randomly within a drivable radius of a center point.
Distances are computed as straight-line approximations (the real app uses OSRM road distances).

In [ ]:
CENTER_LAT =  37.7749
CENTER_LNG = -122.4194
RADIUS_MILES = 8

STATION_NAMES = [
    'Shell', 'Chevron', 'Arco', '76', 'Valero', 'BP', 'Mobil', 'Texaco',
    'Sunoco', 'Marathon', 'Circle K', 'Kwik Trip', 'Casey\'s', 'QuikTrip',
    'Wawa', 'Pilot', 'Love\'s', 'Speedway', 'Holiday', 'GetGo'
]

def haversine_miles(lat1, lng1, lat2, lng2):
    """Great-circle distance in miles between two lat/lng points."""
    R = 3958.8  # Earth radius in miles
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lng2 - lng1)
    a = np.sin(dphi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def generate_stations(n=20, radius_miles=RADIUS_MILES, seed=7):
    np.random.seed(seed)
    lat_deg = radius_miles / 69.0
    lng_deg = radius_miles / (69.0 * np.cos(np.radians(CENTER_LAT)))
    records = []
    while len(records) < n:
        lat = CENTER_LAT + np.random.uniform(-lat_deg, lat_deg)
        lng = CENTER_LNG + np.random.uniform(-lng_deg, lng_deg)
        dist = haversine_miles(CENTER_LAT, CENTER_LNG, lat, lng)
        if dist > radius_miles:
            continue
        records.append({
            'name':             STATION_NAMES[len(records) % len(STATION_NAMES)],
            'lat':              lat,
            'lng':              lng,
            'distance_miles':   round(dist, 3),
            'price_per_gallon': round(np.random.uniform(3.49, 4.59), 3),
        })
    return records

records = generate_stations(n=20)
print(f'Generated {len(records)} stations within {RADIUS_MILES} miles of ({CENTER_LAT}, {CENTER_LNG})')
pd.DataFrame(records).head()

## Attach Effective Cost

Run each synthetic station through the cost engine to compute its effective fill-up cost.

In [ ]:
vehicle = VehicleParams(mpg=28, tank_current=4.5, tank_capacity=13.2)
print(f'Vehicle: {vehicle.mpg} MPG | {vehicle.tank_current} gal in tank | {vehicle.gallons_to_fill:.1f} gal to fill')

station_objs = [
    Station(
        name=r['name'],
        price_per_gallon=r['price_per_gallon'],
        distance_miles=r['distance_miles'],
    )
    for r in records
]

ranked = rank_stations(station_objs, vehicle)

# Merge results back with geo data
result_map = {r.station.name: r for r in ranked}

df = pd.DataFrame([
    {
        **rec,
        'effective_cost':    result_map[rec['name']].effective_cost,
        'savings_vs_best':   result_map[rec['name']].savings_vs_best,
        'rank':              result_map[rec['name']].rank,
        'reachable':         result_map[rec['name']].reachable,
    }
    for rec in records
])

df = df[df['reachable']].copy()
df.sort_values('rank', inplace=True)
print(f'Reachable stations: {len(df)}')
df[['rank', 'name', 'price_per_gallon', 'distance_miles', 'effective_cost']].head(5)

## Cost Map — Price/gal vs Effective Cost

The left map colours stations by **price per gallon** (what a naive user sees).
The right map colours by **effective cost** (what the app recommends).

The gold star marks the recommended station. Notice how it can differ from the cheapest-per-gallon station.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7))
best = df[df['rank'] == 1].iloc[0]
cheapest_pgal = df.loc[df['price_per_gallon'].idxmin()]

def draw_radius(ax, miles):
    # Convert miles to approximate degrees for plotting
    lat_deg = miles / 69.0
    lng_deg = miles / (69.0 * np.cos(np.radians(CENTER_LAT)))
    theta = np.linspace(0, 2 * np.pi, 300)
    ax.plot(CENTER_LNG + lng_deg * np.cos(theta),
            CENTER_LAT + lat_deg * np.sin(theta),
            ls='--', lw=1, color='gray', alpha=0.5)

for ax, col, title, highlight, highlight_label in [
    (axes[0], 'price_per_gallon', 'Naive View: Price per Gallon',
     cheapest_pgal, f'Cheapest/gal: {cheapest_pgal["name"]}'),
    (axes[1], 'effective_cost',   'Smart View: Effective Fill-Up Cost',
     best, f'Recommended: {best["name"]}'),
]:
    sc = ax.scatter(
        df['lng'], df['lat'],
        c=df[col], cmap='RdYlGn_r',
        s=100, zorder=3, edgecolors='white', linewidths=0.5
    )
    plt.colorbar(sc, ax=ax,
                 label='Price/gal ($)' if col == 'price_per_gallon' else 'Effective Cost ($)')

    # User location
    ax.scatter(CENTER_LNG, CENTER_LAT, s=180, color='dodgerblue',
               marker='o', zorder=5, label='Your location')

    # Highlighted pick
    ax.scatter(highlight['lng'], highlight['lat'], s=350,
               color='gold', edgecolors='black', linewidths=1.5,
               marker='*', zorder=6, label=highlight_label)

    # Station name labels for top 5
    for _, row in df.head(5).iterrows():
        ax.annotate(row['name'], (row['lng'], row['lat']),
                    textcoords='offset points', xytext=(5, 4), fontsize=7)

    draw_radius(ax, RADIUS_MILES)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')

plt.suptitle('Gas Station Map — San Francisco Area (Synthetic Data)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(f'Cheapest per gallon: {cheapest_pgal["name"]} (${cheapest_pgal["price_per_gallon"]:.3f}/gal, {cheapest_pgal["distance_miles"]:.1f} mi)')
print(f'Recommended:         {best["name"]} (${best["price_per_gallon"]:.3f}/gal, {best["distance_miles"]:.1f} mi, eff. cost ${best["effective_cost"]:.2f})')

## Savings Map — What the App Saves You

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

# savings_vs_best: 0 for rank 1, negative for all others (they cost more than best)
# Flip sign so positive = costs more than the recommended station
df['extra_cost'] = -df['savings_vs_best']

sc = ax.scatter(
    df['lng'], df['lat'],
    c=df['extra_cost'], cmap='YlOrRd',
    s=120, zorder=3, edgecolors='white', linewidths=0.5
)
plt.colorbar(sc, ax=ax, label='Extra cost vs recommended station ($)')

ax.scatter(CENTER_LNG, CENTER_LAT, s=200, color='dodgerblue',
           marker='o', zorder=5, label='Your location')
ax.scatter(best['lng'], best['lat'], s=400,
           color='#00C853', edgecolors='black', linewidths=1.5,
           marker='*', zorder=6, label=f'Recommended: {best["name"]} ($0 extra)')

for _, row in df.iterrows():
    if row['rank'] <= 5:
        ax.annotate(f'{row["name"]}\n+${row["extra_cost"]:.2f}',
                    (row['lng'], row['lat']),
                    textcoords='offset points', xytext=(5, 4), fontsize=7)

draw_radius(ax, RADIUS_MILES)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('How Much More Each Station Costs vs the Recommended One\n(darker = more expensive choice)', fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print('Full ranking:')
print(df[['rank', 'name', 'price_per_gallon', 'distance_miles', 'effective_cost', 'extra_cost']]
      .rename(columns={'extra_cost': 'extra vs best ($)'})
      .to_string(index=False))